## 1. Basic Tasks

**1. Create a DataFrame from an in-memory Python list/dict and display it.**

In [0]:
data = [
    {"customer_id": 1, "name": "Alice", "loyalty_tier": "Gold"},
    {"customer_id": 2, "name": "Bob", "loyalty_tier": "Silver"},
    {"customer_id": 3, "name": "Charlie", "loyalty_tier": "Bronze"},
    {"customer_id": 4, "name": "Diana", "loyalty_tier": "Platinum"},
    {"customer_id": 5, "name": "Evan", "loyalty_tier": "Silver"},
    {"customer_id": 6, "name": "Fiona", "loyalty_tier": "Gold"},
    {"customer_id": 7, "name": "George", "loyalty_tier": "Bronze"},
    {"customer_id": 8, "name": "Hannah", "loyalty_tier": "Platinum"},
    {"customer_id": 9, "name": "Ian", "loyalty_tier": "Gold"},
    {"customer_id": 10, "name": "Julia", "loyalty_tier": "Silver"},
    {"customer_id": 11, "name": "Kevin", "loyalty_tier": "Bronze"},
    {"customer_id": 12, "name": "Luna", "loyalty_tier": "Silver"}
]

In [0]:
df = spark.createDataFrame(data)
df.display()

**2. Read a CSV with header=True and inferSchema, then read the same file with an explicit StructType
schema; compare the two resulting schemas.**

In [0]:
df1 = spark.read.csv(
    "/Volumes/dev/bronze/raw/sales.csv",
    header=True,
    inferSchema=True
)

In [0]:
from pyspark.sql.types import *

In [0]:
Schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("transaction_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("order_date", DateType(), True)
])

In [0]:
df2 = spark.read.csv(
    "/Volumes/dev/bronze/raw/sales.csv",
    header = True,
    schema = Schema
)

In [0]:
df1.printSchema()
df2.printSchema()
print(df1.schema == df2.schema)

The above comparison returns False but not because of Structure cause both schema have the same column name, same data type and even the same nullability but when we do inferSchema=True
spark autometically detects order_date is a date column and internal metadata `{'__detected_date_formats': 'yyyy-M-d'}`
but the schema i have decleared explicitly has no such metadata. So that's why it returns false. 

**3. Apply one filter transformation followed by one action (count() or show()), and explain in your own
words why nothing ran until the action.**

In [0]:
df2 = df1.filter(df1.order_id == 1)

In [0]:
df2.show()

Apache Spark works on `"Lazy Evaluation"`. So when I am doing any transformation, like `filter()` or `select()`, it dosen't perform any operation, instead it just records the operation and plans out how to actually perform the task. But when I am doing an action like `count()` or `show()` then only it actually performs the task by optimizing the entire plan.

## 2. Intermediate Tasks

**4. Build a small end-to-end ELT: read CSV, filter, add a column (e.g., ingestion_date), and write the
result as Delta.**

In [0]:
df_raw = spark.read.csv(
    "/Volumes/dev/bronze/raw/sales.csv",
    header = True,
    inferSchema = True
)

In [0]:
df_raw = df_raw.dropDuplicates()
df_raw = df_raw.dropna(how = "all")
df_raw = df_raw.dropna(how = "any")

In [0]:
from pyspark.sql.functions import *
df_add_metadata = df_raw.withColumn("ingestion_data", current_timestamp())

In [0]:
df_add_metadata.write.mode("overwrite").saveAsTable("dev.bronze.sales_delta")

**5. Read a JSON file with nested structure and flatten at least one nested field using dot notation or
explode().**

In [0]:
customers_data = [
  {
    "customer_id": 1,
    "name": "Alice",
    "address": {"city": "Seattle", "state": "WA"},
    "orders": ["ORD-001", "ORD-002"]
  },
  {
    "customer_id": 2,
    "name": "Bob",
    "address": {"city": "Austin", "state": "TX"},
    "orders": ["ORD-003"]
  },
  {
    "customer_id": 3,
    "name": "Charlie",
    "address": {"city": "New York", "state": "NY"},
    "orders": ["ORD-004", "ORD-005", "ORD-006"]
  },
  {
    "customer_id": 4,
    "name": "Diana",
    "address": {"city": "Denver", "state": "CO"},
    "orders": ["ORD-007"]
  },
  {
    "customer_id": 5,
    "name": "Evan",
    "address": {"city": "Chicago", "state": "IL"},
    "orders": ["ORD-008", "ORD-009"]
  },
  {
    "customer_id": 6,
    "name": "Fiona",
    "address": {"city": "Miami", "state": "FL"},
    "orders": ["ORD-010"]
  },
  {
    "customer_id": 7,
    "name": "George",
    "address": {"city": "Boston", "state": "MA"},
    "orders": ["ORD-011", "ORD-012"]
  },
  {
    "customer_id": 8,
    "name": "Hannah",
    "address": {"city": "Portland", "state": "OR"},
    "orders": ["ORD-013"]
  },
  {
    "customer_id": 9,
    "name": "Ian",
    "address": {"city": "Phoenix", "state": "AZ"},
    "orders": ["ORD-014", "ORD-015"]
  },
  {
    "customer_id": 10,
    "name": "Julia",
    "address": {"city": "Atlanta", "state": "GA"},
    "orders": ["ORD-016"]
  }
]

In [0]:
df_json = spark.createDataFrame(customers_data)

In [0]:
from pyspark.sql.functions import *
df_flattend = df_json.select(
    col("customer_id"),
    col("name"),
    col("address.city").alias("city"),
    col("address.state").alias("state"),
    explode(col("orders")).alias("single_order")
)

**6. Call .explain() on a multi-step transformation chain and identify, from the physical plan, which steps
got pipelined together versus which required a shuffle.**

In [0]:
df_add_metadata.explain()

## 3. Advanced Tasks

**7. Take a pandas-based script (your own, or a sample provided by your instructor) and rewrite it in
PySpark, documenting at least 3 places where the pandas approach would not scale and how Spark's
approach solves it.**

In [0]:
import pandas as pd

transactions_1 = pd.read_csv("/Volumes/dev/bronze/raw/transaction.csv")
customers_1 = pd.read_csv("/Volumes/dev/bronze/raw/customers-3.csv")

merged_df_1 = pd.merge(transactions, customers, on="customer_id", how="left")

summary_1 = merged_df.groupby("status")["amount"].sum()

summary_1.to_csv("/Volumes/dev/bronze/rawfinancial_summary_1.csv")

In [0]:
from pyspark.sql.functions import sum

transactions_2 = spark.read.csv(
    "/Volumes/dev/bronze/raw/transaction.csv",
    header=True,
    inferSchema=True
)
customers_2 = spark.read.csv(
    "/Volumes/dev/bronze/raw/customers-3.csv",
    header=True,
    inferSchema=True
)

merged_df_2 = transactions_2.join(customers_2, transactions_2.customer_id == customers_2.customer_id, "left")

summary_2 = merged_df_2.groupBy("status").agg(sum("amount"))

summary_2.write.mode("overwrite").csv("/Volumes/dev/bronze/raw/financial_summary_2.csv")

So above I have taken a pandas code implementation and convertated it to the PySpark code. So below I will provide where the pandas implementation would not scale and how the PySpark approach solves it.

**1. Data Ingestion and Memory Allocation**
- Pandas: Pandas read_csv loads the entire dataset into ram of a single machine all at once. If the size of the transactions_1 and customers_1 is more thann the ram size then the system will through `OutOfMemoryError`.
- PySpark: PySpark works on Lazy Evaluation so, first it will only create the plan on how to read it, and later when we do any action then only it will perform the task.
Even when while reading the data it divide the large files into partition and different worker node reads those and if still the memory fills up then spark can spills it to disk, instead of giving error.

**2. Table Joins and Hash Table Overhead**
- Pandas: Pandas merge tries to do the whole join in the memory of a single machine. When joining big tables, it uses a lot of extra memory to match the keys together, which can easily lock up the CPU or crash the system.
- PySpark: PySpark splits the join task across the cluster. It distributes matching customer IDs to different worker nodes so that the memory load is shared. If one table is small, it can also just send a copy of it to all nodes to make the join happen instantly without shuffling data.

**3. Grouping and Aggregations**
- Pandas: When you do groupby in Pandas, it tries to process all the rows one by one using a single processor core. If you have millions of rows, this takes a very long time because it can't share the work.
- PySpark: PySpark does the math in two steps to save time and memory. First, every worker node calculates a partial sum for just its own piece of the data. Then, they only send those tiny summary results to each other to calculate the final total, instead of moving all the raw rows.

**4. Disk Serialization and Output I/O**
- Pandas: Pandas writes the output to the csv file sequentially using just one thread. It writes everything line by line into a single file, which creates a huge bottleneck.
- PySpark: PySpark worker nodes write the output in parallel. Instead of waiting in line to write one big file, all the nodes write their own chunks of data at the exact same time into multiple partitioned files, which is much faster.

**8. Design a partitioning/write strategy (partitionBy, target file sizes) for a table that will mostly be
queried by date range, and justify it using what you know about lazy evaluation and the physical
plan.**

In [0]:
sales_df = spark.read.csv(
    "/Volumes/dev/bronze/raw/sales.csv",
    header=True,
    inferSchema=True
)

cleaned_sales_df = sales_df.dropDuplicates().dropna(how="any")

In [0]:
cleaned_sales_df.write \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .option("maxRecordsPerFile", 10000) \
    .saveAsTable("dev.silver.sales_partition")

In [0]:
%sql
DESC DETAIL dev.silver.sales_partition

In [0]:
%sql
SELECT *
FROM dev.silver.sales_partition
WHERE order_date BETWEEN "2023-12-01" AND "2023-12-02";

Sparks works on lazy evaluation so when I perform any action spark creates a logical plan and then optimize it and then creates a physical plan, and executes the task, so When we perform `partitionBy("order_date")` it partition the table based on order date, and we can check that in the `DESC DETAIL` command that it was partitionBy "order_date".
Also we can define the the maxRecordPerFile using `.option("maxRecordsPerFile", 10000)` this. So instead of reading the entire file at once spark does partition pruning, and read only the required part because I have partitioned by order_date so when I perform this query
```
SELECT *
FROM dev.silver.sales_partition
WHERE order_date BETWEEN "2023-12-01" AND "2023-12-02";
```
spark sql under the hood checks the partitions that was required and not read the entire table, so the speed of the task is increased. So that's how it executes the physical plan.

**9. (Data Analyst) Using a Spark DataFrame (not SQL), reproduce a report you'd normally build in
Excel/pandas — e.g., monthly revenue by category — and export the result for a dashboard.**

In [0]:
from pyspark.sql.functions import sum

transactions = spark.read.csv(
    "/Volumes/dev/bronze/raw/transaction.csv", 
    header=True, 
    inferSchema=True
)
customers = spark.read.csv(
    "/Volumes/dev/bronze/raw/customers-3.csv", 
    header=True, 
    inferSchema=True
)

transactions_clean = transactions.dropDuplicates().dropna()
customers_clean = customers.dropDuplicates().dropna()

merged_df = transactions_clean.join(customers_clean, "customer_id", "left")

summary_df = merged_df.groupBy("status").agg(sum("amount").alias("total_amount"))

summary_df.write.mode("overwrite").saveAsTable("dev.gold.financial_summary")